<a href="https://colab.research.google.com/github/ayaz-ali-36/ai-identification-system/blob/main/Notebooks/day6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install deepface -q

import sqlite3
import json
import numpy as np
from deepface import DeepFace
from sklearn.metrics.pairwise import cosine_similarity
from google.colab.patches import cv2_imshow
from google.colab import files
import cv2
import os

print(" All libraries ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.7/170.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 54.0 MB/s eta 0:00:00
26-06-05 13:26:54 - Directory /root/.deepface has been created
26-06-05 13:26:54 - Directory /root/.deepface/weights has been created
✅ All libraries ready


In [ ]:


# Create database
conn = sqlite3.connect('missing_persons.db')
cursor = conn.cursor()

# Create persons table
cursor.execute('''
    CREATE TABLE IF NOT EXISTS persons (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        age INTEGER,
        gender TEXT,
        location TEXT,
        case_type TEXT DEFAULT 'missing',
        status TEXT DEFAULT 'active',
        photo_path TEXT,
        embedding TEXT,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
''')

# Create search logs table
cursor.execute('''
    CREATE TABLE IF NOT EXISTS search_logs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        searched_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        top_match_id INTEGER,
        confidence_score REAL
    )
''')

conn.commit()
print("Database created ✅")
print("Tables: persons, search_logs")

Database created ✅
Tables: persons, search_logs


In [ ]:
def save_person(name, age, gender, location, case_type, photo_path):
    # Extract embedding from photo
    result = DeepFace.represent(
        img_path=photo_path,
        model_name="Facenet",
        enforce_detection=False
    )
    embedding = result[0]["embedding"]
    embedding_json = json.dumps(embedding)

    # Save to database
    cursor.execute('''
        INSERT INTO persons
        (name, age, gender, location, case_type, photo_path, embedding)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    ''', (name, age, gender, location, case_type, photo_path, embedding_json))

    conn.commit()
    person_id = cursor.lastrowid
    print(f"Saved: {name} → ID {person_id}")
    return person_id

# Upload and save test persons
print("Upload photos one by one:")
uploaded = files.upload()

for filename in uploaded.keys():
    name = filename.split(".")[0].split("_")[0]
    save_person(
        name=name,
        age=25,
        gender="unknown",
        location="Skardu",
        case_type="missing",
        photo_path=filename
    )

Upload photos one by one:


Saving WhatsApp Image 2026-06-04 at 2.35.40 AM.jpeg to WhatsApp Image 2026-06-04 at 2.35.40 AM.jpeg
Saving WhatsApp Image 2026-06-04 at 2.35.41 AM.jpeg to WhatsApp Image 2026-06-04 at 2.35.41 AM.jpeg
Saving WhatsApp Image 2026-06-04 at 3.10.06 AM.jpeg to WhatsApp Image 2026-06-04 at 3.10.06 AM.jpeg
Saving WhatsApp Image 2026-06-04 at 3.27.22 AM.jpeg to WhatsApp Image 2026-06-04 at 3.27.22 AM.jpeg
Saving WhatsApp Image 2026-06-04 at 3.27.23 AM.jpeg to WhatsApp Image 2026-06-04 at 3.27.23 AM.jpeg
Saving WhatsApp Image 2026-06-04 at 3.27.24 AM.jpeg to WhatsApp Image 2026-06-04 at 3.27.24 AM.jpeg
Saving WhatsApp Image 2026-06-04 at 3.34.56 AM.jpeg to WhatsApp Image 2026-06-04 at 3.34.56 AM.jpeg
Saving WhatsApp Image 2026-06-04 at 3.34.58 AM.jpeg to WhatsApp Image 2026-06-04 at 3.34.58 AM.jpeg
Saving WhatsApp Image 2026-06-04 at 3.35.01 AM.jpeg to WhatsApp Image 2026-06-04 at 3.35.01 AM.jpeg
Saving WhatsApp Image 2026-06-04 at 3.35.02 AM.jpeg to WhatsApp Image 2026-06-04 at 3.35.02 AM.jpeg


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/facenet_weights.h5
To: /root/.deepface/weights/facenet_weights.h5
100%|██████████| 92.2M/92.2M [00:00<00:00, 256MB/s]


Saved: WhatsApp Image 2026-06-04 at 2 → ID 1
Saved: WhatsApp Image 2026-06-04 at 2 → ID 2
Saved: WhatsApp Image 2026-06-04 at 3 → ID 3
Saved: WhatsApp Image 2026-06-04 at 3 → ID 4
Saved: WhatsApp Image 2026-06-04 at 3 → ID 5
Saved: WhatsApp Image 2026-06-04 at 3 → ID 6
Saved: WhatsApp Image 2026-06-04 at 3 → ID 7
Saved: WhatsApp Image 2026-06-04 at 3 → ID 8
Saved: WhatsApp Image 2026-06-04 at 3 → ID 9
Saved: WhatsApp Image 2026-06-04 at 3 → ID 10
Saved: WhatsApp Image 2026-06-04 at 3 → ID 11
Saved: WhatsApp Image 2026-06-04 at 3 → ID 12


In [ ]:
def search_face(query_photo_path):
    # Extract embedding from search photo
    result = DeepFace.represent(
        img_path=query_photo_path,
        model_name="Facenet",
        enforce_detection=False
    )
    query_embedding = np.array(result[0]["embedding"]).reshape(1, -1)

    # Load all records from database
    cursor.execute("SELECT id, name, age, location, photo_path, embedding FROM persons")
    all_persons = cursor.fetchall()

    if not all_persons:
        print("Database is empty")
        return []

    results = []
    for person in all_persons:
        person_id, name, age, location, photo_path, embedding_json = person
        db_embedding = np.array(json.loads(embedding_json)).reshape(1, -1)

        from sklearn.metrics.pairwise import cosine_similarity
        score = cosine_similarity(query_embedding, db_embedding)[0][0]

        results.append({
            "id": person_id,
            "name": name,
            "age": age,
            "location": location,
            "photo_path": photo_path,
            "score": round(score * 100, 2)
        })

    # Sort by highest score
    results.sort(key=lambda x: x["score"], reverse=True)

    # Log the search
    if results:
        cursor.execute('''
            INSERT INTO search_logs (top_match_id, confidence_score)
            VALUES (?, ?)
        ''', (results[0]["id"], results[0]["score"]))
        conn.commit()

    return results[:3]

# Upload search photo
print("Upload search photo:")
search_upload = files.upload()
search_photo = list(search_upload.keys())[0]

# Run search
matches = search_face(search_photo)

print("\n=== SEARCH RESULTS ===\n")
for i, match in enumerate(matches):
    label = "HIGH confidence" if match["score"] > 85 else "POSSIBLE match" if match["score"] > 70 else "unlikely"
    print(f"#{i+1}  Name: {match['name']}")
    print(f"     ID: {match['id']}")
    print(f"     Age: {match['age']}")
    print(f"     Location: {match['location']}")
    print(f"     Score: {match['score']}%")
    print(f"     Status: {label}")
    print()

Upload search photo:


Saving WhatsApp Image 2026-06-04 at 3.27.24 AM.jpeg to WhatsApp Image 2026-06-04 at 3.27.24 AM (1).jpeg

=== SEARCH RESULTS ===

#1  Name: WhatsApp Image 2026-06-04 at 3
     ID: 6
     Age: 25
     Location: Skardu
     Score: 100.0%
     Status: HIGH confidence

#2  Name: WhatsApp Image 2026-06-04 at 3
     ID: 5
     Age: 25
     Location: Skardu
     Score: 77.69%
     Status: POSSIBLE match

#3  Name: WhatsApp Image 2026-06-04 at 3
     ID: 4
     Age: 25
     Location: Skardu
     Score: 73.07%
     Status: POSSIBLE match



In [ ]:
print("=== DATABASE CONTENTS ===\n")

cursor.execute("SELECT id, name, age, location, case_type, status, created_at FROM persons")
persons = cursor.fetchall()
print(f"Total persons: {len(persons)}\n")
for p in persons:
    print(f"ID:{p[0]} | {p[1]} | Age:{p[2]} | {p[3]} | {p[4]} | {p[5]} | {p[6]}")

print("\n=== SEARCH LOGS ===\n")
cursor.execute("SELECT * FROM search_logs")
logs = cursor.fetchall()
print(f"Total searches: {len(logs)}\n")
for log in logs:
    print(f"Search ID:{log[0]} | Top match ID:{log[2]} | Score:{log[3]}% | {log[1]}")

=== DATABASE CONTENTS ===

Total persons: 12

ID:1 | WhatsApp Image 2026-06-04 at 2 | Age:25 | Skardu | missing | active | 2026-06-05 13:29:03
ID:2 | WhatsApp Image 2026-06-04 at 2 | Age:25 | Skardu | missing | active | 2026-06-05 13:29:07
ID:3 | WhatsApp Image 2026-06-04 at 3 | Age:25 | Skardu | missing | active | 2026-06-05 13:29:08
ID:4 | WhatsApp Image 2026-06-04 at 3 | Age:25 | Skardu | missing | active | 2026-06-05 13:29:10
ID:5 | WhatsApp Image 2026-06-04 at 3 | Age:25 | Skardu | missing | active | 2026-06-05 13:29:11
ID:6 | WhatsApp Image 2026-06-04 at 3 | Age:25 | Skardu | missing | active | 2026-06-05 13:29:15
ID:7 | WhatsApp Image 2026-06-04 at 3 | Age:25 | Skardu | missing | active | 2026-06-05 13:29:17
ID:8 | WhatsApp Image 2026-06-04 at 3 | Age:25 | Skardu | missing | active | 2026-06-05 13:29:20
ID:9 | WhatsApp Image 2026-06-04 at 3 | Age:25 | Skardu | missing | active | 2026-06-05 13:29:22
ID:10 | WhatsApp Image 2026-06-04 at 3 | Age:25 | Skardu | missing | active | 202